In [ ]:
# Check GPU availability
!nvidia-smi

In [ ]:
# Install required packages
print("📦 Installing dependencies...")
!pip install -q gradio transformers peft accelerate
print("✅ Dependencies installed!")

In [ ]:
# Import required libraries
import gradio as gr
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import json
import re
from typing import Dict, Tuple
import os

print("✅ Libraries imported successfully!")
print(f"📊 PyTorch version: {torch.__version__}")
print(f"📊 CUDA available: {torch.cuda.is_available()}")

In [ ]:
# Configuration for Kaggle - AUTO-DETECT MODEL PATH
# ============================================================================
# 🎯 This cell automatically finds your fine-tuned model (searches recursively)
# ============================================================================

import os
from pathlib import Path

BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"      # Base model (will be downloaded)
MODEL_PATH = None

# Auto-detect model path
print("🔍 Searching for fine-tuned model...")

if os.path.exists("/kaggle/input"):
    print(f"📂 Scanning /kaggle/input/ recursively...\n")
    
    # Search recursively for adapter_config.json or adapter_model.safetensors
    for root, dirs, files in os.walk("/kaggle/input"):
        # Check if this directory contains model files
        if "adapter_config.json" in files or "adapter_model.safetensors" in files:
            MODEL_PATH = root
            print(f"✅ Found fine-tuned model at: {MODEL_PATH}")
            print(f"📂 Model files found:")
            for f in sorted(files)[:15]:
                print(f"   ✓ {f}")
            if len(files) > 15:
                print(f"   ... and {len(files) - 15} more files")
            break
    
    if MODEL_PATH is None:
        print("❌ Could not find fine-tuned model!")
        print("\n📂 Here's what I found in /kaggle/input/:")
        for root, dirs, files in os.walk("/kaggle/input"):
            level = root.replace("/kaggle/input", "").count(os.sep)
            indent = "   " * level
            print(f"{indent}{os.path.basename(root)}/")
            sub_indent = "   " * (level + 1)
            for file in files[:5]:
                print(f"{sub_indent}- {file}")
            if len(files) > 5:
                print(f"{sub_indent}... and {len(files) - 5} more files")
        
        print("\n📌 TROUBLESHOOTING:")
        print("   1. Make sure you clicked 'Add Data' and added your model dataset")
        print("   2. Required files: adapter_config.json, adapter_model.safetensors")
        print("   3. The dataset should be visible in the right sidebar under 'DATASETS'")
        print("   4. Re-run this cell after adding data")
        
        # Don't raise error, allow manual path setting
        print("\n⚠️ Attempting to use default path anyway...")
        MODEL_PATH = "/kaggle/input/fine-tuned-model"
else:
    # Fallback for local testing
    MODEL_PATH = "./fine_tuned_model"
    print(f"⚠️ Not on Kaggle. Using local path: {MODEL_PATH}")

print(f"\n📊 Final Configuration:")
print(f"   Model path: {MODEL_PATH}")
print(f"   Base model: {BASE_MODEL}")
print(f"   Device: {'CUDA (GPU)' if torch.cuda.is_available() else 'CPU'}")
print("\n✅ Configuration complete! Proceed to next cell.")

In [ ]:
# Define the LogClassifier class
class LogClassifier:
    """Main classifier for analyzing logs and detecting MITRE ATT&CK techniques."""
    
    def __init__(self, model_path: str, base_model: str):
        """Initialize the log classifier."""
        self.model_path = model_path
        self.base_model = base_model
        self.model = None
        self.tokenizer = None
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        
    def load_model(self):
        """Load the fine-tuned model and tokenizer."""
        print(f"🔄 Loading model from {self.model_path}...")
        print(f"📊 Using device: {self.device}")
        
        # Load base model
        print(f"   Loading base model: {self.base_model}")
        base_model = AutoModelForCausalLM.from_pretrained(
            self.base_model,
            torch_dtype=torch.float16 if self.device == "cuda" else torch.float32,
            device_map="auto" if self.device == "cuda" else None,
            trust_remote_code=True
        )
        
        # Load LoRA adapters
        print(f"   Loading LoRA adapters from: {self.model_path}")
        self.model = PeftModel.from_pretrained(base_model, self.model_path)
        self.model.eval()
        
        # Load tokenizer
        print(f"   Loading tokenizer...")
        self.tokenizer = AutoTokenizer.from_pretrained(
            self.base_model,
            trust_remote_code=True
        )
        self.tokenizer.pad_token = self.tokenizer.eos_token
        
        print(f"✅ Model loaded successfully!")
        if self.device == "cuda":
            print(f"📊 GPU Memory: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
        
        return True
    
    def format_prompt(self, log_input: str, instruction: str = None) -> str:
        """Format the input into the correct prompt structure."""
        if instruction is None:
            instruction = "Analyze this session log chunk and determine if it contains normal or suspicious activity. If suspicious, identify all MITRE ATT&CK techniques and explain why."
        
        prompt = f"""{instruction}

### Input:
{log_input}

### Response:
"""
        return prompt
    
    def parse_output(self, output: str) -> Dict[str, str]:
        """Parse the model output to extract status, reason, and MITRE techniques."""
        result = {
            "status": "Unknown",
            "reason": "",
            "mitre_techniques": [],
            "raw_output": output
        }
        
        # Try to parse as JSON first (some models output JSON)
        try:
            # Look for JSON in the output
            json_match = re.search(r'\{[\s\S]*?"result"[\s\S]*?\}', output)
            if json_match:
                json_data = json.loads(json_match.group())
                if "result" in json_data:
                    res = json_data["result"]
                    # Handle different JSON formats
                    if "normal" in res:
                        result["status"] = "Normal" if res["normal"] else "Suspicious"
                    if "suspicious" in res:
                        result["status"] = "Suspicious" if res["suspicious"] else "Normal"
                    if "reason" in res:
                        result["reason"] = res["reason"]
                    if "mitre_techniques" in res:
                        result["mitre_techniques"] = res["mitre_techniques"]
        except:
            pass  # If JSON parsing fails, continue with text parsing
        
        # Text-based status extraction - try multiple patterns
        if result["status"] == "Unknown":
            status_patterns = [
                r'Status:\s*(Normal|Suspicious)',
                r'Status\s*[-:]\s*(Normal|Suspicious)',
                r'(Normal|Suspicious)\s*activity',
                r'This\s+(?:is|appears|looks)\s+(normal|suspicious)',
                r'"normal"\s*:\s*(true|false)',
                r'"suspicious"\s*:\s*(true|false)',
            ]
            
            for pattern in status_patterns:
                status_match = re.search(pattern, output, re.IGNORECASE)
                if status_match:
                    matched = status_match.group(1).lower()
                    if matched == "true":
                        result["status"] = "Normal"
                    elif matched == "false":
                        result["status"] = "Suspicious"
                    else:
                        result["status"] = status_match.group(1).capitalize()
                    break
        
        # Keyword-based fallback
        if result["status"] == "Unknown":
            output_lower = output.lower()
            if "suspicious" in output_lower or "attack" in output_lower or "malicious" in output_lower:
                result["status"] = "Suspicious"
            elif "normal" in output_lower or "legitimate" in output_lower or "benign" in output_lower or "no malicious" in output_lower:
                result["status"] = "Normal"
        
        # Extract reason if not already found
        if not result["reason"]:
            reason_patterns = [
                r'Reason:\s*(.+?)(?:\n\n|\Z)',
                r'Reason\s*[-:"]\s*(.+?)(?:\n\n|"|\Z)',
                r'Explanation:\s*(.+?)(?:\n\n|\Z)',
                r'Analysis:\s*(.+?)(?:\n\n|\Z)',
            ]
            
            for pattern in reason_patterns:
                reason_match = re.search(pattern, output, re.DOTALL | re.IGNORECASE)
                if reason_match:
                    result["reason"] = reason_match.group(1).strip().replace('"', '')
                    break
        
        # Fallback: extract text after JSON or first meaningful paragraph
        if not result["reason"]:
            # Remove JSON part
            clean_output = re.sub(r'\{[\s\S]*?\}', '', output).strip()
            # Get first paragraph or sentences
            sentences = clean_output.split('\n')
            meaningful = [s.strip() for s in sentences if len(s.strip()) > 20]
            if meaningful:
                result["reason"] = ' '.join(meaningful[:3])[:500]
            elif output.strip():
                result["reason"] = output.strip()[:500]
        
        # Extract MITRE ATT&CK techniques  
        mitre_pattern = r'T\d{4}(?:\.\d{3})?(?:\s*[-:]\s*[A-Za-z\s]+)?'
        mitre_matches = re.findall(mitre_pattern, output)
        result["mitre_techniques"] = [m.strip() for m in mitre_matches if m.strip()]
        
        return result
    
    def classify_log(
        self,
        log_input: str,
        max_new_tokens: int = 512,
        temperature: float = 0.7,
        top_p: float = 0.9
    ) -> Tuple[str, str, str, str]:
        """
        Classify a log entry and return status, reason, and MITRE techniques.
        
        Returns:
            Tuple of (status, reason, mitre_techniques, raw_output)
        """
        if not self.model:
            return "Error", "Model not loaded. Please load the model first.", "", ""
        
        if not log_input.strip():
            return "Error", "Please provide log input.", "", ""
        
        # Validate JSON if it looks like JSON
        if log_input.strip().startswith('{'):
            try:
                json.loads(log_input)
            except json.JSONDecodeError as e:
                return "Error", f"Invalid JSON: {str(e)}", "", ""
        
        # Format prompt
        prompt = self.format_prompt(log_input)
        
        # Tokenize
        inputs = self.tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=3072
        )
        inputs = {k: v.to(self.model.device) for k, v in inputs.items()}
        
        # Generate
        print(f"🚀 Generating prediction...")
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                do_sample=True,
                top_p=top_p,
                pad_token_id=self.tokenizer.eos_token_id
            )
        
        # Decode
        generated_text = self.tokenizer.decode(
            outputs[0][inputs['input_ids'].shape[1]:],
            skip_special_tokens=True
        )
        
        print(f"📝 Generated {len(generated_text)} characters")
        print(f"📝 Preview: {generated_text[:300]}...")
        
        # Parse output
        result = self.parse_output(generated_text)
        
        # Format MITRE techniques for display
        mitre_display = "\n".join([f"• {tech}" for tech in result["mitre_techniques"]]) if result["mitre_techniques"] else "None detected"
        
        return (
            result["status"],
            result["reason"],
            mitre_display,
            result["raw_output"]
        )

print("✅ LogClassifier class defined!")

In [ ]:
# Initialize the classifier
classifier = LogClassifier(model_path=MODEL_PATH, base_model=BASE_MODEL)

# UI Helper Functions
def load_model_ui():
    """Load model button handler."""
    try:
        classifier.load_model()
        return "✅ Model loaded successfully! You can now classify logs."
    except Exception as e:
        return f"❌ Error loading model: {str(e)}"

def classify_log_ui(log_input, max_tokens, temperature, top_p):
    """Classify log button handler."""
    try:
        status, reason, mitre, raw = classifier.classify_log(
            log_input,
            max_new_tokens=int(max_tokens),
            temperature=float(temperature),
            top_p=float(top_p)
        )
        
        # Format status with emoji
        status_emoji = "🟢" if status == "Normal" else "🔴" if status == "Suspicious" else "⚠️"
        status_display = f"{status_emoji} {status}"
        
        return status_display, reason, mitre, raw
    except Exception as e:
        return f"❌ Error", f"Error during classification: {str(e)}", "", ""

print("✅ Helper functions defined!")

In [ ]:
# Example logs for testing
EXAMPLE_NORMAL = """{
  "metadata": {
    "session_id": "20251026_012500",
    "chunk_index": 1,
    "start_time": "2025-10-25T19:30:00.000Z",
    "end_time": "2025-10-25T19:30:05.000Z",
    "number_of_events": 3
  },
  "logs": [
    {
      "timestamp": "2025-10-25T19:30:00.554Z",
      "event_type": "network",
      "summary": "Ether / IP / TCP 10.10.13.2:443 > 192.168.1.100:54321 A",
      "layers": {
        "IP": {"src": "10.10.13.2", "dst": "192.168.1.100"},
        "TCP": {"sport": "443", "dport": "54321", "flags": "A"}
      }
    }
  ]
}"""

EXAMPLE_SUSPICIOUS = """{
  "metadata": {
    "session_id": "20251026_attack",
    "chunk_index": 42,
    "start_time": "2025-10-26T14:30:00.000Z",
    "end_time": "2025-10-26T14:30:05.000Z",
    "number_of_events": 5
  },
  "logs": [
    {
      "timestamp": "2025-10-26T14:30:01.000Z",
      "event_type": "process",
      "EventID": 4688,
      "Process": "powershell.exe",
      "CommandLine": "powershell.exe -EncodedCommand SQBuAHYAbwBrAGUALQBXAGUAYgBSAGUAcQB1AGUAcwB0AA==",
      "User": "SYSTEM",
      "ParentProcess": "cmd.exe"
    },
    {
      "timestamp": "2025-10-26T14:30:02.000Z",
      "event_type": "network",
      "summary": "Ether / IP / TCP 10.10.13.2:54321 > 45.33.32.156:4444 S",
      "layers": {
        "IP": {"src": "10.10.13.2", "dst": "45.33.32.156"},
        "TCP": {"sport": "54321", "dport": "4444", "flags": "S"}
      }
    }
  ]
}"""

print("✅ Example logs loaded!")

In [ ]:
# Create Gradio Interface
demo = gr.Blocks(title="MITRE ATT&CK Log Classifier", theme=gr.themes.Soft())

with demo:
    gr.Markdown(
        """
        # 🔍 MITRE ATT&CK Log Classifier
        ### Analyze system logs for suspicious activity and identify MITRE ATT&CK techniques
        
        This tool uses a fine-tuned AI model to analyze logs and detect potential security threats.
        """
    )
    
    with gr.Row():
        with gr.Column():
            load_btn = gr.Button("🚀 Load Model", variant="primary", size="sm")
            load_status = gr.Textbox(label="Model Status", lines=1, interactive=False)
    
    with gr.Row():
        with gr.Column(scale=2):
            log_input = gr.Textbox(
                label="📋 Log Input (JSON format)",
                placeholder="Paste your log data here in JSON format...",
                lines=15,
                value=EXAMPLE_NORMAL
            )
            
            with gr.Accordion("⚙️ Advanced Settings", open=False):
                max_tokens = gr.Slider(
                    minimum=128,
                    maximum=1024,
                    value=512,
                    step=64,
                    label="Max New Tokens"
                )
                temperature = gr.Slider(
                    minimum=0.1,
                    maximum=1.0,
                    value=0.7,
                    step=0.1,
                    label="Temperature"
                )
                top_p = gr.Slider(
                    minimum=0.1,
                    maximum=1.0,
                    value=0.9,
                    step=0.1,
                    label="Top-p (Nucleus Sampling)"
                )
            
            classify_btn = gr.Button("🔍 Classify Log", variant="primary", size="lg")
            
            with gr.Row():
                gr.Examples(
                    examples=[
                        [EXAMPLE_NORMAL],
                        [EXAMPLE_SUSPICIOUS]
                    ],
                    inputs=[log_input],
                    label="📚 Example Logs"
                )
        
        with gr.Column(scale=2):
            status_output = gr.Textbox(
                label="🎯 Status",
                lines=1,
                interactive=False
            )
            reason_output = gr.Textbox(
                label="💡 Reason",
                lines=6,
                interactive=False
            )
            mitre_output = gr.Textbox(
                label="🎯 MITRE ATT&CK Techniques",
                lines=4,
                interactive=False
            )
            
            # Make Raw Output ALWAYS visible and BIGGER (not in accordion)
            raw_output = gr.Textbox(
                label="📄 Full Model Response (Raw Output)",
                lines=15,
                max_lines=30,
                interactive=False,
                show_copy_button=True
            )
    
    gr.Markdown(
        """
        ---
        ### 📖 How to Use:
        1. **Load Model**: Click "Load Model" to initialize the AI model (takes 1-2 minutes, one-time step)
        2. **Input Log**: Paste your log data in JSON format or use example logs
        3. **Classify**: Click "Classify Log" to analyze the log
        4. **Review Results**: Check the Status, Reason, and identified MITRE ATT&CK techniques
        
        ### 📝 Input Format:
        - Accepts JSON-formatted logs with metadata and events
        - Can handle network logs, system logs, process creation events, etc.
        - See examples for reference format
        
        ### 🎯 Output:
        - **Status**: Normal or Suspicious
        - **Reason**: Explanation of the classification
        - **MITRE Techniques**: Identified attack techniques (e.g., T1105 - Ingress Tool Transfer)
        - **Full Model Response**: Complete raw output from the model (useful for debugging)
        """
    )
    
    # Event handlers
    load_btn.click(
        fn=load_model_ui,
        inputs=[],
        outputs=[load_status]
    )
    
    classify_btn.click(
        fn=classify_log_ui,
        inputs=[log_input, max_tokens, temperature, top_p],
        outputs=[status_output, reason_output, mitre_output, raw_output]
    )

print("✅ Gradio interface created!")

In [ ]:
# Launch the interface
print("="*80)
print("🚀 Starting MITRE ATT&CK Log Classifier UI")
print("="*80)
print("\n📋 Instructions:")
print("   1. Click 'Load Model' button above to initialize the AI model")
print("   2. Wait for model to load (takes 1-2 minutes)")
print("   3. Paste your log data or use the example logs")
print("   4. Click 'Classify Log' to analyze")
print("\n🌐 The interface will appear below and provide a public URL")
print("="*80)
print()

# Launch with share=True to get a public URL on Kaggle
demo.launch(share=True, debug=True)

---

## 📌 Important Notes for Kaggle

### First-Time Setup:
1. **Upload Your Model**: 
   - Go to Kaggle → Datasets → New Dataset
   - Upload your `fine_tuned_model` folder
   - Name it: `fine-tuned-model`

2. **Add Dataset to Notebook**:
   - Click "Add Data" (right sidebar)
   - Search for your dataset
   - Click "Add"

3. **Enable Internet** (for downloading base model):
   - Settings → Internet → ON

### What Happens When You Run:

**Cell 1-3**: Setup and install dependencies  
**Cell 4**: Configure paths (verify model exists)  
**Cell 5**: Define the classifier class  
**Cell 6**: Initialize classifier and helper functions  
**Cell 7**: Load example logs  
**Cell 8**: Create Gradio UI  
**Cell 9**: Launch the UI (⭐ **RUN THIS LAST**)

### After Launch:

- You'll see a **public Gradio URL** (e.g., `https://xxxxx.gradio.live`)
- Click it to access your UI from anywhere
- Share the link with others (it expires after 72 hours)

### Model Loading Time:

- **First load**: ~2-3 minutes (downloads base model ~3GB)
- **Subsequent loads**: ~30 seconds (uses cache)

### Troubleshooting:

**"Model path not found"**:
- Make sure you added the dataset in the "Add Data" section
- Check the path in Cell 4 matches your dataset name

**"CUDA out of memory"**:
- Kaggle provides free T4 GPU with 15GB VRAM
- This model should fit comfortably (~4GB)
- If issues, try: Settings → Accelerator → GPU T4 x2

**"Can't download base model"**:
- Enable Internet in Settings
- Kaggle blocks some downloads by default

---

## 🎉 You're All Set!

Run all cells in order, then click the Gradio link to access your log classifier UI!